In [45]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset

In [57]:
PATH='data'
df_train = pd.read_csv(f'{PATH}/df_train.csv')
df_test = pd.read_csv(f'{PATH}/df_test.csv')

In [61]:
sorted(df_train.label.unique()),sorted(df_test.label.unique())

([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])

In [46]:
df = pd.read_csv('data.csv')
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

In [9]:
df = df.dropna(subset=['text'])

In [55]:
num_labels = len(label_encoder.classes_)
df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_encoded']
)
train_dataset = Dataset.from_pandas(df_train[['text', 'label_encoded']].rename(columns={'label_encoded': 'label'}))
test_dataset = Dataset.from_pandas(df_test[['text', 'label_encoded']].rename(columns={'label_encoded': 'label'}))

In [56]:
df_test

,text,label,label_encoded
23749,fanbox,Мобильные услуги,5
1883,Почему уменьшается баланс.,Баланс,3
22380,как проверить карту ЕКО?,Оплата,7
7386,Что я подключил,Баланс,3
16429,можете написать тарифы только для общения и их...,тарифы - подбор,13
...,...,...,...
25216,Установлен тариф0 почему платить приходится 10...,мобильная связь - тарифы,12
21038,как мне узнать мой номер?,SIM-карта и номер,2
30543,Куда были потрачены единицы?,Баланс,3
26263,Архив тариф,мобильная связь - тарифы,12


In [19]:
df_train.label.value_counts(normalize=True)

label
мобильная связь - тарифы               0.365613
Мобильные услуги                       0.125578
FAQ - тарифы и услуги                  0.112623
Баланс                                 0.090610
SIM-карта и номер                      0.081973
тарифы - подбор                        0.058471
Мобильный интернет                     0.041426
Оплата                                 0.036611
FAQ - интернет                         0.027783
Личный кабинет                         0.019376
Устройства                             0.015554
Роуминг                                0.011579
запрос обратной связи                  0.007758
мобильная связь - зона обслуживания    0.005045
Name: proportion, dtype: float64

In [20]:
df_test.label.value_counts(normalize=True)

label
мобильная связь - тарифы               0.365637
Мобильные услуги                       0.125650
FAQ - тарифы и услуги                  0.112657
Баланс                                 0.090645
SIM-карта и номер                      0.081932
тарифы - подбор                        0.058392
Мобильный интернет                     0.041425
Оплата                                 0.036686
FAQ - интернет                         0.027667
Личный кабинет                         0.019260
Устройства                             0.015592
Роуминг                                0.011617
запрос обратной связи                  0.007796
мобильная связь - зона обслуживания    0.005044
Name: proportion, dtype: float64

In [11]:
MODEL_NAME = "cointegrated/rubert-tiny2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/26167 [00:00<?, ? examples/s]

Map:   0%|          | 0/6542 [00:00<?, ? examples/s]

In [36]:
from torch import nn
from transformers import Trainer
from sklearn.utils.class_weight import compute_class_weight

# 1. Считаем веса классов на основе обучающей выборки
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_train['label_encoded']),
    y=df_train['label_encoded']
)
class_weights = class_weights.astype(np.float32)
class_weights = torch.from_numpy(class_weights)

# 2. Создаем кастомный Trainer, который учитывает эти веса
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Считаем базовый лосс без весов (он считается на MPS)
        loss_fct = nn.CrossEntropyLoss(reduction='none')
        raw_loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        # --- МАГИЯ СИНХРОНИЗАЦИИ УСТРОЙСТВ ---
        # Переносим метки текущего батча на CPU, чтобы безопасно достать нужные веса
        labels_cpu = labels.view(-1).to("cpu")
        
        # Достаем веса классов на CPU
        batch_weights_cpu = class_weights[labels_cpu]
        
        # Перекидываем готовые веса батча на то устройство, где крутится модель (MPS)
        batch_weights = batch_weights_cpu.to(raw_loss.device)
        # -------------------------------------
        
        # Перемножаем и берем среднее
        weighted_loss = (raw_loss * batch_weights).mean()
        
        return (weighted_loss, outputs) if return_outputs else weighted_loss

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

training_args = TrainingArguments(
    output_dir="./results",          
    learning_rate=2e-5,              
    per_device_train_batch_size=32,  
    per_device_eval_batch_size=32,
    num_train_epochs=3,              
    weight_decay=0.01,
    eval_strategy="epoch",     
    save_strategy="epoch",           
    load_best_model_at_end=True,     
    logging_steps=10,
    report_to="none"                 
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,1.429600,1.492484,0.631917
2,1.341300,1.156587,0.673036
3,0.954600,1.073503,0.697952


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=2454, training_loss=1.4753651362468094, metrics={'train_runtime': 376.1024, 'train_samples_per_second': 208.722, 'train_steps_per_second': 6.525, 'total_flos': 144947100068352.0, 'train_loss': 1.4753651362468094, 'epoch': 3.0})

In [ ]:
predictions = trainer.predict(tokenized_test)
preds_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

target_names = [str(cls) for cls in label_encoder.classes_]

print(classification_report(true_labels, preds_labels, target_names=target_names))

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)



--- Финальный отчет BERT ---
                                     precision    recall  f1-score   support

                     FAQ - интернет       0.00      0.00      0.00       181
              FAQ - тарифы и услуги       0.78      0.63      0.70       737
                  SIM-карта и номер       0.81      0.95      0.87       536
                             Баланс       0.76      0.82      0.79       593
                     Личный кабинет       0.97      0.23      0.37       126
                   Мобильные услуги       0.65      0.74      0.69       822
                 Мобильный интернет       0.51      0.85      0.64       271
                             Оплата       0.81      0.67      0.73       240
                            Роуминг       0.67      0.43      0.53        76
                         Устройства       0.89      0.39      0.54       102
              запрос обратной связи       0.00      0.00      0.00        51
мобильная связь - зона обслуживания       0.0

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

In [24]:
df['label'].value_counts(normalize=True)

label
мобильная связь - тарифы               0.365618
Мобильные услуги                       0.125592
FAQ - тарифы и услуги                  0.112630
Баланс                                 0.090617
SIM-карта и номер                      0.081965
тарифы - подбор                        0.058455
Мобильный интернет                     0.041426
Оплата                                 0.036626
FAQ - интернет                         0.027760
Личный кабинет                         0.019352
Устройства                             0.015561
Роуминг                                0.011587
запрос обратной связи                  0.007765
мобильная связь - зона обслуживания    0.005044
Name: proportion, dtype: float64

In [41]:
df[df['label']=='мобильная связь - зона обслуживания'].head(5).text

516     Сделайте 4G в Апкане!!!!
832            4g в селе байтик?
921               карта покрытия
1193         как пользоваться 4g
1293               Зона покрытия
Name: text, dtype: object

In [49]:
df.iloc[516]

text                        Сделайте 4G в Апкане!!!!
label            мобильная связь - зона обслуживания
label_encoded                                     11
Name: 516, dtype: object

In [42]:
df[df['label']=='запрос обратной связи'].text

114                    Низкая скорость интернета
115                    Низкая скорость интернета
159                                      Спасибо
232                        Хочу заказать звонок 
260                         Оставить жалобу хочу
                          ...                   
32109    Хочу оставить заявку на обратный звонок
32245                            Запрос «услуга»
32325                              Перехожу на О
32460                                Позвони мне
32574                            Запрос «услуга»
Name: text, Length: 254, dtype: object

In [43]:
df[df['label']=='FAQ - интернет'].text

6               Как запретить все звонки вне сети оператор
20                              Ручной настройки интернета
101      И ещё могу вам видео отправить на WhatsApp ещё...
102                 Мне нужен тариф выгодный для интернета
137           Ха держать Интернет 2G за что я вам плачу а?
                               ...                        
32450               Отправь мне ручные настройки интернета
32523                                   Ватсап номер барбы
32528                          Отправьте мне автонастройки
32556                    Снимется ли плата за видео звонок
32611                            Неделный платеж по тарифу
Name: text, Length: 908, dtype: object

In [44]:
model.save_pretrained("models/fine_tuned_rubert_tiny2")
tokenizer.save_pretrained("models/fine_tuned_rubert_tiny2")

('models/fine_tuned_rubert_tiny2/tokenizer_config.json',
 'models/fine_tuned_rubert_tiny2/special_tokens_map.json',
 'models/fine_tuned_rubert_tiny2/vocab.txt',
 'models/fine_tuned_rubert_tiny2/added_tokens.json',
 'models/fine_tuned_rubert_tiny2/tokenizer.json')

In [47]:
import joblib
joblib.dump(label_encoder, 'models/label_encoder.pkl')

['models/label_encoder.pkl']